# Day 6 Assignment — Silver Layer

## Task 2 — Create Silver Table

The Silver layer is created from the Bronze sales table.

The Bronze data is cleaned by:
- Removing duplicate records
- Fixing column data types
- Removing records with invalid quantities
- Removing records with invalid monetary values
- Removing records with invalid or missing order dates

The cleaned data is stored in:

`dev.silver.sales_clean1`

In [0]:
bronze_df = spark.table("dev.bronze.sales_raw1")
display(bronze_df)

In [0]:
from pyspark.sql.functions import sum, when, col

### Check data quality first

In [0]:
bronze_df.select(
    *[
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in bronze_df.columns
    ]
).display()

### Remove exact duplicate records

In [0]:
silver_df = bronze_df.dropDuplicates()

In [0]:
print("Bronze records:", bronze_df.count())
print("After duplicate removal:", silver_df.count())

### Fix the data types

In [0]:
from pyspark.sql.functions import to_date

In [0]:
silver_df = (
    silver_df
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("transaction_id", col("transaction_id").cast("int"))
    .withColumn("product_id", col("product_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
)

In [0]:
silver_df.printSchema()

### Remove invalid records

In [0]:
silver_df = (
    silver_df
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("transaction_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("quantity").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("discount_amount").isNotNull())
    .filter(col("discount_amount") >= 0)
    .filter(col("total_amount").isNotNull())
    .filter(col("total_amount") >= 0)
    .filter(col("order_date").isNotNull())
)

### Remove duplicate transactions

In [0]:
silver_df = silver_df.dropDuplicates(["transaction_id"])

In [0]:
print("Final Silver record count:", silver_df.count())

In [0]:
silver_df.write.mode("overwrite").saveAsTable("dev.silver.sales_clean1")

## Silver Layer Result

The Bronze sales data was transformed into a cleaned Silver table.

The following transformations were performed:

1. Removed duplicate records.
2. Converted ID columns to integer data types.
3. Converted `quantity` to integer.
4. Converted `discount_amount` and `total_amount` to double.
5. Converted `order_date` from string to date.
6. Removed records with missing required IDs.
7. Removed records where quantity was less than or equal to zero.
8. Removed records with negative discount amounts.
9. Removed records with negative total amounts.
10. Removed records with missing order dates.
11. Removed duplicate `transaction_id` values.

The final cleaned table is:

`dev.silver.sales_clean1`